# Promise Integrity: analysis walkthrough

Run after the pipeline. This notebook uses real aggregate outputs and a read-only warehouse connection. Costs and ETA revisions remain scenarios. See DATA_LICENSE.md for Olist attribution.

In [1]:
from pathlib import Path
import json
import pandas as pd
import duckdb
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
s = json.loads((ROOT / 'reports/summary.json').read_text())
con = duckdb.connect(str(ROOT / 'data/warehouse.duckdb'), read_only=True)
print('Eligible orders:', s['eligible_orders'])

Eligible orders: 96470


## 1. Verify the grain
A row must represent one order before calculating order rates or merchandise totals.

In [2]:
print(con.execute('SELECT COUNT(*) AS order_rows, COUNT(DISTINCT order_id) unique_orders, SUM(item_value_brl) item_value_brl FROM mart_orders').df().to_string(index=False))

 order_rows  unique_orders  item_value_brl
      99441          99441    1.359164e+07


## 2. Reproduce the main finding
The denominator is reviewed on-time orders, not all orders. Missing reviews do not imply satisfaction.

In [3]:
print(con.execute('''SELECT COUNT(*) AS reviewed_on_time, SUM(poor_review::INT) AS hidden_failures, AVG(poor_review::INT) AS rate FROM mart_orders WHERE eligible AND on_time AND poor_review IS NOT NULL''').df().to_string(index=False))

 reviewed_on_time  hidden_failures     rate
            89151           8238.0 0.092405


## 3. Find high-volume investigation targets
A category rate describes an association. It does not identify the root cause.

In [4]:
print(pd.read_csv(ROOT / 'reports/categories.csv').head(6).to_string(index=False))

              segment  reviewed_on_time  hidden_failures     rate   ci_low  ci_high
       bed_bath_table              8262              958 0.115953 0.109227 0.123035
computers_accessories              6011              673 0.111961 0.104237 0.120182
      furniture_decor              5586              601 0.107590 0.099733 0.115988
        health_beauty              7881              572 0.072580 0.067057 0.078518
       sports_leisure              6908              504 0.072959 0.067061 0.079332
        watches_gifts              5013              479 0.095552 0.087721 0.104002


## 4. Review timing sensitivity
Explain why coverage differs so much between on-time and late orders before comparing their review rates.

In [5]:
print(pd.read_csv(ROOT / 'reports/review_sensitivity.csv').to_string(index=False))
print('Bootstrap gap:', s['bootstrap_gap'])

  group  eligible_orders  post_delivery_review_coverage  post_delivery_poor_rate  any_timing_review_n  any_timing_poor_rate
on_time            89936                       0.991272                 0.092405                89443              0.092606
   late             6534                       0.291552                 0.194226                 6381              0.623570
Bootstrap gap: {'difference': 0.10182070108955195, 'ci_low': 0.08137218041373391, 'ci_high': 0.12512860550020474, 'iterations': 1000, 'method': 'purchase-week cluster bootstrap; association, not causation'}


## 5. Test a mix explanation
Common-state standardization does not produce a sign reversal here. Do not invent Simpson's paradox just to demonstrate a technique.

In [6]:
print(json.dumps(s['standardization'], indent=2))

{
  "crude_single": 0.07505926388022457,
  "crude_multi": 0.2468614598377958,
  "standardized_single": 0.0748710478597771,
  "standardized_multi": 0.24800714352666936,
  "common_states": 18,
  "sign_reversal": false,
  "note": "Common-state direct standardization; no causal interpretation. Different populations for crude and standardized estimates."
}


## 6. Evaluate the delay model
Compare precision-recall evidence and assumed costs, not just ROC AUC. All preprocessing and threshold choices precede the test evaluation.

In [7]:
print(pd.DataFrame(s['model']['metrics']).to_string(index=False))
print('Split sizes:', s['model']['splits'])
print('Alert threshold:', s['model']['threshold'])

              model  roc_auc  average_precision  brier_score  assumed_error_cost_brl  alerts  precision   recall
           logistic  0.82814           0.301861     0.016879                  4768.0     803   0.120797 0.477833
training_prevalence  0.50000           0.017881     0.019309                  5075.0       0        NaN 0.000000
Split sizes: {'train': 60309, 'validation': 11422, 'test': 11353}
Alert threshold: 0.14


## 7. Re-score the metric without changing the service
The ETA scenario holds delivery timestamps fixed. Cost sensitivity is not a measurement of profit.

In [8]:
print(pd.read_csv(ROOT / 'reports/eta_scenarios.csv').to_string(index=False))
print(pd.read_csv(ROOT / 'reports/cost_scenarios.csv').to_string(index=False))
con.close()

 extension_days_assumed  orders  recorded_on_time_rate  rescored_on_time_rate  reclassified_orders      evidence
                      0   96470               0.932269               0.932269                    0 scenario_only
                      1   96470               0.932269               0.940821                  825 scenario_only
                      2   96470               0.932269               0.946470                 1370 scenario_only
                      3   96470               0.932269               0.951653                 1870 scenario_only
                      5   96470               0.932269               0.960983                 2770 scenario_only
                      7   96470               0.932269               0.970333                 3672 scenario_only
 margin_assumed  recovery_brl_assumed  hidden_failure_orders  recovery_exposure_brl  negative_proxy_orders  proxy_contribution_brl         evidence
            0.1                    10                   8238 

## Exercises
1. Add a state filter and recompute the correct denominator.
2. Explain why the review timing rule changes the late-order population.
3. Identify five forbidden future features in a handoff-time model.
4. Change an assumed cost and explain what business evidence would be needed to validate it.
5. Run the test suite before changing eligibility logic.